# Part 2: Train + Validate — Sarah's First Logistic Regression
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Wednesday — Training the Model

Yesterday Sarah built the preprocessor. Today she chains it with a logistic regression and trains her first model.

She has three questions to answer before walking into Marcus's office on Friday:

1. **Does the model learn anything useful?** (or is it no better than guessing "stayed" for everyone?)
2. **Which features actually matter?** (Marcus will ask — and Sarah needs to know.)
3. **Is the accuracy number trustworthy?** (One split gives one number. Is that number stable?)

**By the end of this notebook you will be able to:**
- Chain preprocessing + model into a single sklearn `Pipeline`
- Train with one `.fit()` call and predict with one `.predict()` call
- Read the trained coefficients to explain which features drive churn predictions
- Use 5-fold cross-validation to get a stable accuracy estimate

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — including sklearn.linear_model.LogisticRegression")

## Step 1 — Rebuild the dataset split and preprocessor

We re-run the same steps from Tuesday so this notebook is self-contained. Reading the dataset in 4 lines, splitting, and defining the preprocessor.

In [ ]:
# Load and split
df = pd.read_csv("data/northstar_churn.csv")
y  = df["churned"]
X  = df.drop(columns=["customer_id", "churned"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,
)

numeric_features = [
    "age", "tenure_months", "num_purchases_quarter",
    "avg_monthly_spend_gbp", "returns_per_purchase",
    "last_login_days_ago", "avg_review_polarity",
    "support_tickets_quarter",
]
categorical_features = ["region", "subscription_tier"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

print(f"Training set: {len(X_train):,} customers")
print(f"Test set:     {len(X_test):,} customers")
print("Preprocessor ready (median impute + standard scale | mode impute + one-hot)")

## Step 2 — Build the full pipeline: preprocessor + model

The whole project becomes ONE object. `.fit()` trains everything end-to-end; `.predict()` applies every transformation in the right order. No leakage, no manual juggling.

In [ ]:
# Logistic regression — most interpretable classifier; right starting baseline.
# max_iter raised because the default sometimes warns about non-convergence on this dataset.
model = LogisticRegression(max_iter=1000, random_state=42)

# Wrap preprocessor + model in one Pipeline
full_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", model),
])

print("✅ Full pipeline built.")
print()
print("Steps in order:")
for name, step in full_pipeline.steps:
    print(f"  · {name:8s}  →  {step.__class__.__name__}")

## ⏸️ Pause and Predict

Before we train, predict for yourself: which features will get the strongest POSITIVE coefficients (push toward 'churned'), and which will get the strongest NEGATIVE coefficients (push toward 'stayed')?

Some hints from Monday's exploration:
- Short tenure → high churn
- Premium subscribers → likely lower churn
- High return rate, many support tickets → probably high churn
- Negative review polarity → probably high churn

*Write your top-3 positive and top-3 negative predictions here:*

> *Sample expected:*
> - **Positive (push toward churn):** returns_per_purchase, support_tickets_quarter, last_login_days_ago
> - **Negative (push toward stay):** tenure_months, subscription_tier_premium, avg_review_polarity

## Step 3 — Fit on training data

This is the line where the model actually learns. The preprocessor learns its parameters (medians, modes, category lists, scaling means/stds) and the logistic regression learns its coefficients.

In [ ]:
full_pipeline.fit(X_train, y_train)

print("✅ Pipeline fitted.")
print()
print(f"Training accuracy:   {full_pipeline.score(X_train, y_train):.3f}")
print(f"Test accuracy:       {full_pipeline.score(X_test,  y_test):.3f}")
print()
print(f"Baseline (always 'stayed'): {1 - y_test.mean():.3f}  ← anything better than this is learning something.")

### 💡 What you should notice

- **Test accuracy is a couple of points above baseline.** The model has learned *something* — but on an imbalanced dataset (88% baseline), a few percentage points of accuracy could be statistical noise. We need a better metric than accuracy to judge this model. That's tomorrow's notebook.
- **Training accuracy and test accuracy are similar.** That's a sign the model is NOT overfitting. (If training was much higher than test, we'd worry.)
- **The next obvious question is "WHY does the model predict what it predicts?"** Logistic regression makes this answerable — let's inspect the coefficients.

## Step 4 — Coefficient inspection

Each feature gets one coefficient (`β`). Sign tells you direction; magnitude tells you strength (because features are scaled to comparable units after preprocessing).

> **Positive β** — higher feature value → higher predicted probability of churn.
> **Negative β** — higher feature value → lower predicted probability of churn.

In [ ]:
# Grab the trained logistic regression model and the feature names
feature_names = full_pipeline.named_steps["prep"].get_feature_names_out()
coefs         = full_pipeline.named_steps["model"].coef_[0]

# Build a tidy dataframe sorted by absolute strength
coef_df = pd.DataFrame({
    "feature":     [name.split("__")[-1] for name in feature_names],
    "coefficient": coefs,
    "abs_coef":    np.abs(coefs),
}).sort_values("abs_coef", ascending=False).reset_index(drop=True)

print("Top features ranked by absolute coefficient size:")
print(coef_df[["feature", "coefficient"]].head(10).to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = coef_df.head(12).iloc[::-1]
colors  = ["coral" if c > 0 else "steelblue" for c in plot_df["coefficient"]]
ax.barh(plot_df["feature"], plot_df["coefficient"], color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Coefficient (positive = pushes toward CHURN)")
ax.set_title("Logistic regression coefficients — top 12 by magnitude")
plt.tight_layout()
plt.show()

### 💡 What you should notice

- **`returns_per_purchase` has the strongest positive coefficient.** High return rate → higher predicted churn probability. Matches intuition: people who return a lot are unhappy.
- **`avg_review_polarity` has a strong negative coefficient.** Higher (more positive) reviews → lower predicted churn. Also matches intuition.
- **`tenure_months` is negative.** Longer-tenured customers churn less. Matches yesterday's tenure-bucket chart.
- **`subscription_tier_premium` is negative**, while `_basic` and `_free` are positive or near zero — premium customers churn less.

**Back to our scenario:**
> Sarah can now answer Marcus's question "why does the model predict what it predicts?" — pointing at this chart and saying "returns and reviews are the strongest signals, followed by tenure and subscription tier." That's interpretability.

## Step 5 — 5-fold cross-validation

The single 80/20 split gives ONE accuracy number. Is that number stable, or did we get lucky? `cross_val_score` re-runs the train-and-evaluate cycle 5 times on different folds of the training data, giving us a mean and standard deviation.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    full_pipeline,
    X_train, y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

print(f"5-fold CV accuracy scores: {[f'{s:.3f}' for s in cv_scores]}")
print()
print(f"Mean: {cv_scores.mean():.3f}")
print(f"Std:  {cv_scores.std():.3f}")
print()
print(f"Stable estimate: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print()
print(f"Versus baseline (always 'stayed'): {1 - y_train.mean():.3f}")
print()
print("→ Cross-validated accuracy is a much more honest number than the single split.")
print("→ But we already know accuracy isn't the right metric. Tomorrow: precision and recall.")

## ✅ Section Summary

| What we did | Result |
|---|---|
| **Built the full pipeline** | One sklearn `Pipeline` — preprocess + LogisticRegression — fits and predicts in one call |
| **Fitted on training data** | The line where the model actually learns its parameters |
| **Inspected coefficients** | `returns_per_purchase` (+), `avg_review_polarity` (−), `tenure_months` (−) drive predictions — all match intuition |
| **5-fold CV accuracy** | Stable around `~88-89%` — but accuracy ≈ baseline because the data is imbalanced |

**Key insight for our scenario:**
> Sarah has a trained model and can explain WHY it predicts what it predicts (via the coefficients). But the accuracy number is suspicious — it's barely above baseline. Tomorrow she has to look at the confusion matrix to see whether the model is actually *catching* churners — or just predicting "stayed" most of the time.

---
**Up next → Part 3:** Thursday — confusion matrix, precision, recall, F1, and the business decision about where to set the classification threshold.
Open `04_metrics_threshold.ipynb`

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## Extension 1 — Why `stratify=y` matters in CV too

`StratifiedKFold` ensures each fold has roughly the same churn rate as the full dataset. Without stratification, you could get unlucky — one fold with 5% churn rate and another with 18%, making accuracy comparisons across folds meaningless.

You can verify by checking each fold's distribution:

In [ ]:
for i, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    fold_churn_rate = y_train.iloc[val_idx].mean()
    print(f"Fold {i+1} validation churn rate: {fold_churn_rate:.1%}")
print()
print("→ All within 1% of each other. Stratification is doing its job.")

## Extension 2 — Regularization with the `C` parameter

`LogisticRegression(C=1.0)` is the default. `C` is the *inverse* of regularization strength — smaller `C` = stronger regularization = simpler model with smaller coefficients = less variance, more bias.

Let's compare three values.

In [ ]:
for C_val in [0.01, 1.0, 100.0]:
    pipe = Pipeline([
        ("prep",  preprocessor),
        ("model", LogisticRegression(C=C_val, max_iter=1000, random_state=42)),
    ])
    score = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy",
                             n_jobs=-1).mean()
    coefs = pipe.fit(X_train, y_train).named_steps["model"].coef_[0]
    print(f"C = {C_val:>6}  CV acc = {score:.3f}  Max |coef| = {np.abs(coefs).max():.3f}")

### 💡 What this tells us

- **Strong regularization (C=0.01)** — coefficients are shrunk toward zero. The model is simpler. Sometimes this generalises better on small or noisy datasets.
- **Weak regularization (C=100)** — coefficients are bigger. The model fits training data more aggressively. Risk of overfitting on more complex data, but on our clean simulated dataset the effect is small.
- **For this dataset**, all three values give similar CV accuracy. Regularization mostly matters in higher-dimensional or noisier real-world problems.

## Extension 3 — Comparing logistic regression to a baseline classifier

How much value is the model really adding? Compare to a `DummyClassifier` that just predicts the majority class.

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy_score = cross_val_score(dummy, X_train, y_train, cv=cv, scoring="accuracy",
                              n_jobs=-1).mean()

print(f"Dummy classifier (always 'stayed'): {dummy_score:.3f}")
print(f"Our logistic regression:              {cv_scores.mean():.3f}")
print(f"Lift from logistic regression:        +{cv_scores.mean() - dummy_score:.3f}")
print()
print("On accuracy alone the lift is tiny — about 1 percentage point. Because the data is imbalanced,")
print("ANY classifier that learns SOMETHING will score similar to baseline. We need real metrics now.")